In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch_geometric.datasets as datasets
import networkx as nx
from torch_geometric.nn import GCNConv

In [ ]:
# 加载数据集
#dataset = datasets.Actor(root='data/Actor')
#dataset = datasets.Amazon(root='data/AmazonComputers', name='Computers')
#dataset = datasets.Amazon(root='data/AmazonPhoto', name='Photo')
#dataset = datasets.AttributedGraphDataset(root='data/BlogCatalog',name='BlogCatalog')
#dataset = datasets.Planetoid(root='data/CiteSeer', name='CiteSeer')
#dataset = datasets.Coauthor(root='data/CoauthorCS', name='CS')
#dataset = datasets.Coauthor(root='data/CoauthorPhy', name='Physics')
#dataset = datasets.Planetoid(root='data/Cora', name='Cora')
#dataset = datasets.WebKB(root='data/Cornell',name='Cornell')
#dataset = datasets.CitationFull(root='data/DBLP',name='DBLP')
#dataset = datasets.AttributedGraphDataset(root='data/facebook',name='facebook')
#dataset = datasets.HeterophilousGraphDataset(root='data/Minesweeper',name='Minesweeper')
#dataset = datasets.NELL(root='data/NELL')
#dataset = datasets.Planetoid(root='data/pubmed',name='pubmed')
#dataset = datasets.WebKB(root='data/Texas',name='Texas')
#dataset = datasets.AttributedGraphDataset(root='data/Wiki',name='Wiki')
#dataset = datasets.WebKB(root='data/Wisconsin',name='Wisconsin')
dataset = datasets.KarateClub()
file_name = 'KarateClub'

data = dataset[0]
uniform_x = torch.ones_like(data.x)
print(uniform_x.shape)


In [ ]:
# GCN结构框架
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)
    
class ProjectionHead(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(in_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, hidden_channels),
        )
        
    def forward(self, x):
        return self.mlp(x)

In [ ]:
def info_nce_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    sim_matrix = torch.mm(z1, z2.t()) / temperature

    labels = torch.arange(z1.size(0))
    loss_1 = F.cross_entropy(sim_matrix, labels)
    loss_2 = F.cross_entropy(sim_matrix.t(), labels)
    return (loss_1 + loss_2) / 2


In [ ]:
gcn = GCN(len(data.x[1]), 128, 64)
self = GCN(len(uniform_x[1]), 128, 64)
proj_head_gcn = ProjectionHead(64, 64, 64)
proj_head_self = ProjectionHead(64, 64, 64)

optimizer = torch.optim.Adam(list(gcn.parameters()) + list(self.parameters()), lr=0.01)

In [ ]:
for epoch in range(1, 201):
    gcn.train()
    self.train()
    proj_head_gcn.train()
    proj_head_self.train()
    
    optimizer.zero_grad()

    h1 = gcn(data.x, data.edge_index)
    z1 = proj_head_gcn(h1)

    h2 = self(uniform_x, data.edge_index)
    z2 = proj_head_self(h2)
    
    loss = info_nce_loss(z1, z2)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.8f}')

In [ ]:
gcn.eval()
self.eval()
with torch.no_grad():
    z_s = gcn(data.x, data.edge_index)
    z_f = self(uniform_x, data.edge_index)
    diff_score = torch.norm(z_s - z_f, dim=1)

In [ ]:
node_ids = torch.arange(data.num_nodes)
importance_list = list(zip(node_ids.numpy(), diff_score.numpy()))

importance_sorted = sorted(importance_list, key=lambda x: x[1], reverse=True)

print("Top 10 important nodes:")
for rank, (node_id, score) in enumerate(importance_sorted[:10], 1):
    print(f"Rank {rank}: Node {node_id}, Score: {score:.4f}")